# Despliegue de Aplicaciones LLM en Producción

> Aprende las mejores prácticas para desplegar sistemas LLM en producción:
> contenedores, variables de entorno, health checks y escalado.

## Objetivos
- Dockerizar una aplicación FastAPI + Claude
- Gestionar secretos y configuración de producción
- Implementar health checks y graceful shutdown
- Desplegar en Cloud Run, Railway o Fly.io

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic fastapi uvicorn python-dotenv --quiet

## 2. Arquitectura de despliegue

In [ ]:
print("""=== ARQUITECTURA DE DESPLIEGUE LLM EN PRODUCCIÓN ===

Opciones de despliegue (de menor a mayor complejidad):

1. SERVERLESS (más sencillo):
   - Vercel / Netlify Functions
   - AWS Lambda + API Gateway
   - Google Cloud Functions
   → Ideal para: tráfico variable, sin estado
   → Límite: timeout corto, sin conexiones persistentes

2. CONTENEDORES MANAGED (recomendado):
   - Google Cloud Run
   - Railway.app
   - Fly.io
   - Render.com
   → Ideal para: APIs con FastAPI/Flask
   → Ventaja: escala a cero, fácil CI/CD

3. KUBERNETES (alta escala):
   - GKE, EKS, AKS
   - Self-managed K3s
   → Ideal para: miles de req/s, SLA alto
   → Complejo de operar

Stack recomendado para empezar:
  FastAPI + Docker → Cloud Run + Secret Manager + Cloud Monitoring
""")

## 3. Aplicación FastAPI lista para producción

In [ ]:
APP_CODE = '''
import os
import time
import logging
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import anthropic

# Configuración desde variables de entorno
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]  # Nunca hardcodear
MODELO = os.getenv("MODELO", "claude-haiku-4-5-20251001")
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "500"))
ENTORNO = os.getenv("ENTORNO", "desarrollo")

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s")
logger = logging.getLogger(__name__)

# Cliente global (reutilizar conexión)
cliente_anthropic: anthropic.Anthropic = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Inicializar recursos al arrancar, limpiar al parar."""
    global cliente_anthropic
    logger.info(f"Iniciando app en entorno: {ENTORNO}")
    cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    logger.info("Cliente Anthropic inicializado")
    yield
    logger.info("Apagando app (graceful shutdown)")

app = FastAPI(
    title="LLM API",
    version="1.0.0",
    lifespan=lifespan,
    docs_url="/docs" if ENTORNO != "produccion" else None,  # Deshabilitar Swagger en prod
)

app.add_middleware(CORSMiddleware,
    allow_origins=["https://tudominio.com"],  # No usar * en producción
    allow_methods=["POST", "GET"],
    allow_headers=["*"])

class SolicitudChat(BaseModel):
    mensaje: str = Field(..., min_length=1, max_length=4000)
    system_prompt: str = Field(default="Eres un asistente útil.", max_length=1000)

class RespuestaChat(BaseModel):
    respuesta: str
    tokens_entrada: int
    tokens_salida: int
    latencia_ms: int

@app.get("/health")
async def health_check():
    """Health check para Kubernetes/Cloud Run."""
    return {"status": "ok", "entorno": ENTORNO, "modelo": MODELO}

@app.get("/ready")
async def readiness_check():
    """Readiness: verifica que el cliente está inicializado."""
    if cliente_anthropic is None:
        raise HTTPException(status_code=503, detail="Cliente no inicializado")
    return {"status": "ready"}

@app.post("/chat", response_model=RespuestaChat)
async def chat(solicitud: SolicitudChat, request: Request):
    inicio = time.perf_counter()
    logger.info(f"Chat request de {request.client.host}, {len(solicitud.mensaje)} chars")
    try:
        resp = cliente_anthropic.messages.create(
            model=MODELO,
            max_tokens=MAX_TOKENS,
            system=solicitud.system_prompt,
            messages=[{"role": "user", "content": solicitud.mensaje}]
        )
        latencia_ms = int((time.perf_counter() - inicio) * 1000)
        logger.info(f"Respuesta en {latencia_ms}ms, {resp.usage.output_tokens} tokens salida")
        return RespuestaChat(
            respuesta=resp.content[0].text,
            tokens_entrada=resp.usage.input_tokens,
            tokens_salida=resp.usage.output_tokens,
            latencia_ms=latencia_ms
        )
    except anthropic.RateLimitError:
        raise HTTPException(status_code=429, detail="Rate limit alcanzado")
    except anthropic.APIError as e:
        logger.error(f"Error Anthropic API: {e}")
        raise HTTPException(status_code=502, detail="Error del proveedor LLM")
'''

with open("app_produccion.py", "w", encoding="utf-8") as f:
    f.write(APP_CODE)
print("✓ app_produccion.py generado")

## 4. Dockerfile y docker-compose

In [ ]:
DOCKERFILE = """FROM python:3.11-slim

# Usuario no-root para seguridad
RUN useradd -m -u 1000 appuser
WORKDIR /app

# Instalar dependencias primero (caché de Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar código
COPY --chown=appuser:appuser . .
USER appuser

# Health check integrado
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

EXPOSE 8000
CMD ["uvicorn", "app_produccion:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

DOCKER_COMPOSE = """version: '3.8'
services:
  llm-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - ANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}  # Desde .env
      - ENTORNO=produccion
      - MODELO=claude-haiku-4-5-20251001
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  nginx:  # Proxy inverso opcional
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf
    depends_on:
      - llm-api
"""

REQUIREMENTS = """anthropic>=0.40.0
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
python-dotenv>=1.0.0
pydantic>=2.0.0
"""

ENV_EXAMPLE = """# .env.example — NO commitear .env con valores reales
ANTHROPIC_API_KEY=sk-ant-XXXX
ENTORNO=desarrollo
MODELO=claude-haiku-4-5-20251001
MAX_TOKENS=500
"""

for nombre, contenido in [
    ("Dockerfile", DOCKERFILE),
    ("docker-compose.yml", DOCKER_COMPOSE),
    ("requirements.txt", REQUIREMENTS),
    (".env.example", ENV_EXAMPLE),
]:
    with open(nombre, "w") as f:
        f.write(contenido)
    print(f"✓ {nombre} generado")

print("""
Para ejecutar localmente:
  cp .env.example .env && nano .env   # Añadir tu API key
  docker-compose up --build
  curl http://localhost:8000/health
""")

## 5. CI/CD y despliegue en Cloud Run

In [ ]:
GITHUB_ACTIONS = """# .github/workflows/deploy.yml
name: Deploy to Cloud Run

on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Autenticar en Google Cloud
        uses: google-github-actions/auth@v2
        with:
          credentials_json: ${{ secrets.GCP_SA_KEY }}

      - name: Build y push imagen
        run: |
          gcloud builds submit --tag gcr.io/${{ secrets.GCP_PROJECT }}/llm-api

      - name: Desplegar en Cloud Run
        run: |
          gcloud run deploy llm-api \\
            --image gcr.io/${{ secrets.GCP_PROJECT }}/llm-api \\
            --region europe-west1 \\
            --platform managed \\
            --allow-unauthenticated \\
            --set-secrets ANTHROPIC_API_KEY=anthropic-key:latest \\
            --set-env-vars ENTORNO=produccion \\
            --min-instances 0 \\
            --max-instances 10 \\
            --memory 512Mi \\
            --concurrency 80
"""

import os
os.makedirs(".github/workflows", exist_ok=True)
with open(".github/workflows/deploy.yml", "w") as f:
    f.write(GITHUB_ACTIONS)
print("✓ .github/workflows/deploy.yml generado")

print("""
=== CHECKLIST DE PRODUCCIÓN ===

Seguridad:
  ✓ API key en Secret Manager, nunca en código
  ✓ CORS restringido a dominios propios
  ✓ Dockerfile con usuario no-root
  ✓ Rate limiting implementado
  ✓ Validación de inputs con Pydantic

Observabilidad:
  ✓ Logging estructurado
  ✓ Health check (/health) y readiness (/ready)
  ✓ Métricas de latencia y tokens
  ✓ Alertas en Cloud Monitoring / Datadog

Escalado:
  ✓ Contenedor stateless (escala horizontalmente)
  ✓ min-instances > 0 para evitar cold starts en producción
  ✓ Graceful shutdown con lifespan de FastAPI
  ✓ Límites de memoria y CPU definidos

CI/CD:
  ✓ Tests automatizados antes del deploy
  ✓ Deploy automático solo a main
  ✓ Rollback fácil con versiones de Cloud Run
""")